In [1]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm
from scipy import stats

from plotnine import *
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [2]:
lofcorr_df = pl.read_parquet("/home/dnanexus/data_dir/association_files/loftee_correlation_quantitative_EUR_genebass1e6coding_maf1e3.parquet")

lofcorr_df.filter(pl.col('top_plof_var_count')>50)

region,gene_name,phenotype,n_variants,n_plof,n_variants_nonull,loftee_corr,n_individuals_plof,top_plof_var,top_plof_var_count,plof_prop,t_stat,corr_pval
str,str,str,u64,i64,u64,f32,list[u32],str,u32,f64,f64,f64
"""ENSG00000143921""","""ABCG8""","""seated_height_int""",999,67,999,0.02261,"[88, 26, … 1]","""chr2:43873809:C:T""",88,0.067067,0.714092,0.475337
"""ENSG00000152254""","""G6PC2""","""forced_expiratory_volume_in_1s…",247,29,247,0.003034,"[52, 35, … 1]","""chr2:168902461:CG:C""",52,0.117409,0.047484,0.962166
"""ENSG00000146477""","""SLC22A3""","""total_bilirubin_int""",460,38,460,-0.03182,"[55, 32, … 1]","""chr6:160451026:G:GA""",55,0.082609,-0.681318,0.496015
"""ENSG00000204305""","""AGER""","""whole_body_fat_mass_int""",340,48,340,0.055098,"[257, 223, … 1]","""chr6:32183113:C:A""",257,0.141176,1.0145,0.31107
"""ENSG00000164181""","""ELOVL7""","""platelet_crit_int""",183,23,183,0.043998,"[347, 182, … 1]","""chr5:60767867:G:A""",347,0.125683,0.592506,0.554251
…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000143375""","""CGN""","""hip_circumference_int""",935,108,935,-0.047025,"[448, 145, … 1]","""chr1:151530732:C:T""",448,0.115508,-1.437983,0.150774
"""ENSG00000100983""","""GSS""","""calcium_int""",306,26,306,-0.077198,"[70, 62, … 1]","""chr20:34932046:G:A""",70,0.084967,-1.350014,0.178016
"""ENSG00000168769""","""TET2""","""whole_body_water_mass_int""",2182,689,2182,0.018486,"[396, 196, … 1]","""chr4:105234920:AT:A""",396,0.315765,0.863271,0.388083


In [5]:
# Read the full Genebass results
pval_cutoffs = {'burden': 6.7e-7, 'skato': 2.5e-7}

# Convert the pandas operations into a single Polars expression
gb_pl = (
    pl.read_parquet('/home/dnanexus/data_dir/association_files/genebass_all_associations_p_e-5.parquet')
    .with_columns(
        significant = (pl.col('Pvalue') < pval_cutoffs['skato']) | (pl.col('Pvalue_Burden') < pval_cutoffs['burden']),
        phenotype = pl.col('description').str.replace_all(r"\(", "").str.replace_all(r"\)", "").str.replace_all(' ', '_').str.to_lowercase() + '_int',
        region = pl.col('gene_id'),
        gene_name = pl.col('gene_symbol'),
    )
    .filter(
        (pl.col('significant') == True) &
        (pl.col('modifier') != 'custom') &
        # (pl.col('annotation').str.contains('pLoF'))
        (pl.col('annotation') == 'pLoF')
    )
    .select(['region', 'gene_name', 'phenotype', 'trait_type'])
    .unique()
)

# Keep the top hit per gene
# gb_pl = gb_pl.sort("Pvalue").unique(subset=["region"], keep="first")

gene_trait_df = gb_pl[['region', 'gene_name', 'phenotype', 'trait_type']].unique()

plof = pl.read_parquet('/home/dnanexus/data_dir/association_files/rvat_EUR_500k_regenie.parquet').with_columns(
    phenotype = (pl.col('trait') + '_int'),
    region = pl.col('gene_id'),
    rvat_pval = (10** -pl.col("neg_log10p")),
).filter(
    (pl.col('trait_type') == 'quantitative')
)

lofcorr_df = pl.read_parquet("/home/dnanexus/data_dir/association_files/loftee_correlation_quantitative_EUR_genebass1e6coding_maf1e3.parquet")

gene_trait_df = gb_pl.join(plof[['region', 'phenotype', 'beta', 'rvat_pval']], on=['region', 'phenotype'], how='inner')
gene_trait_df = gene_trait_df.join(lofcorr_df, on=['region', 'gene_name', 'phenotype'], how='left').with_columns(
    sign_agreement = (pl.col('loftee_corr')*pl.col('beta') > 0) & (pl.col('loftee_corr').is_not_nan()),
    corr_dir = (pl.col('loftee_corr')/pl.col('loftee_corr').abs())
)

# TODO: We don't cover all genebass significant genes, so we need to filter out nans (LOFTEE correlations missing)
# gene_trait_df = gene_trait_df.filter(pl.col('sign_agreement')==1)
gene_trait_df

region,gene_name,phenotype,trait_type,beta,rvat_pval,n_variants,n_plof,n_variants_nonull,loftee_corr,n_individuals_plof,top_plof_var,top_plof_var_count,plof_prop,t_stat,corr_pval,sign_agreement,corr_dir
str,str,str,str,f64,f64,u64,i64,u64,f32,list[u32],str,u32,f64,f64,f64,bool,f32
"""ENSG00000068976""","""PYGM""","""lymphocyte_percentage_int""","""continuous""",0.0320271,0.547635,771,51,771,0.031337,"[89, 50, … 1]","""chr11:64753958:CG:C""",89,0.066148,0.869438,0.384879,true,1.0
"""ENSG00000173575""","""CHD2""","""lymphocyte_percentage_int""","""continuous""",0.0271693,0.525649,957,17,957,0.280234,"[510, 2, … 1]","""chr15:92992955:AATGCAGGAATACG:…",510,0.017764,9.021569,9.9693e-19,true,1.0
"""ENSG00000196712""","""NF1""","""lymphocyte_percentage_int""","""continuous""",-0.792415,6.3358e-19,1605,129,1605,-0.218112,"[24, 9, … 1]","""chr17:31232069:TCAGAG:T""",24,0.080374,-8.94808,9.7538e-19,true,-1.0
"""ENSG00000004939""","""SLC4A1""","""lymphocyte_percentage_int""","""continuous""",-0.861268,0.000003,672,23,672,-0.211524,"[3, 2, … 1]","""chr17:44259808:C:T""",3,0.034226,-5.601924,3.0966e-8,true,-1.0
"""ENSG00000101745""","""ANKRD12""","""lymphocyte_percentage_int""","""continuous""",-0.482395,2.7033e-16,1462,124,1462,-0.147921,"[55, 25, … 1]","""chr18:9256020:AAGAG:A""",55,0.084815,-5.714913,1.3285e-8,true,-1.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000112077""","""RHAG""","""reticulocyte_count_int""","""continuous""",0.952411,1.8038e-41,315,16,315,0.174576,"[149, 5, … 1]","""chr6:49636655:C:T""",149,0.050794,3.136737,0.001871,true,1.0
"""ENSG00000029534""","""ANK1""","""reticulocyte_count_int""","""continuous""",0.656849,2.1682e-10,1392,34,1392,0.269449,"[18, 12, … 1]","""chr8:41672574:CTG:C""",18,0.024425,10.431616,1.3985e-24,true,1.0
"""ENSG00000197969""","""VPS13A""","""reticulocyte_count_int""","""continuous""",0.176348,2.1188e-7,2047,225,2047,0.035972,"[109, 67, … 1]","""chr9:77339540:C:CT""",109,0.109917,1.627776,0.103726,true,1.0


In [6]:
from anngeno import AnnGeno

# --- 2. Read anngeno and get genotypes ---
ag = AnnGeno('/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag', low_mem=True)

phenotype = 'ldl_direct_int'
gene_id = 'ENSG00000130164'
gdata = ag.get_many_regions([gene_id])[gene_id]

plof_burdens = pl.DataFrame({
    'individual': ag.samples,
    'burden': gdata['genotypes'].T @ gdata['annotations']['loftee_hc'].to_numpy()
})#.filter(pl.col('burden') < 2)

config_path = f'/home/dnanexus/ukbgym/config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)
cov_list = config.get("covariates", [])

# --- 2. Read and merge data ---
pheno_path = '/home/dnanexus/data_dir/phenotypes/phenotypes190_missing20_unique2_int.parquet'
prs_path = '/home/dnanexus/data_dir/phenotypes/PRS190_EUR_missing20_unique2_int.parquet'
cov_path = '/home/dnanexus/data_dir/phenotypes/250709_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet'

pheno_lazy = pl.scan_parquet(pheno_path).select("IID", phenotype).rename({"IID": "individual"}).with_columns(pl.col("individual").cast(pl.String)) # Cast to String
prs_lazy = pl.scan_parquet(prs_path).select("IID", f"{phenotype}_prs").rename({"IID": "individual"}).with_columns(pl.col("individual").cast(pl.String)) # Cast to String
cov_lazy = pl.scan_parquet(cov_path).select("sample", *cov_list).rename({"sample": "individual"})

# Join everything and collect into a pandas DataFrame for the regression
merged_pd = (
    pheno_lazy
    .join(prs_lazy, on="individual", how="inner")
    .join(cov_lazy, on="individual", how="inner")
    .drop_nulls()
    .drop_nans()
    .collect()
    .to_pandas()
)

# --- 3. Compute residuals for the single phenotype ---
# Define the dependent (y) and independent (X) variables
y = merged_pd[phenotype]
X = merged_pd[[f"{phenotype}_prs"] + cov_list]
X = sm.add_constant(X) # Add an intercept

# Fit the linear model and get the residuals
model = sm.OLS(y, X).fit()
residuals = model.resid

# --- 4. Create the final long-format DataFrame ---
p_df = pl.DataFrame({
    "individual": merged_pd["individual"].astype(str),
    "phenotype": phenotype,
    "pheno_value": residuals
})

# 4. Join with gene burdens
plot_gp_df = plof_burdens.join(p_df, on='individual', how='inner')
plot_gp_df

/home/dnanexus/anngeno/anngeno/anngeno.py:14: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
/home/dnanexus/deeprvat2-env/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
/home/dnanexus/deeprvat2-env/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.


individual,burden,phenotype,pheno_value
str,i8,str,f64
"""3834985""",0,"""ldl_direct_int""",-0.016527
"""1649233""",0,"""ldl_direct_int""",0.400111
"""5689749""",0,"""ldl_direct_int""",-0.761904
"""1160905""",0,"""ldl_direct_int""",2.267405
"""5102203""",0,"""ldl_direct_int""",0.03693
…,…,…,…
"""2864227""",0,"""ldl_direct_int""",-0.894074
"""5733954""",0,"""ldl_direct_int""",0.548599
"""3923515""",0,"""ldl_direct_int""",-0.737069
